In [ ]:
import re
import nltk
import random
import numpy as np
import pandas as pd
# import seaborn as sns
# import matplotlib.pyplot as plt

try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')
import torch.nn as nn
from sklearn.metrics import f1_score
from gensim.models import Word2Vec
from collections import Counter
# Setting as large the xtick and ytick font sizes in graphs

# plt.rcParams['xtick.labelsize'] = 'large'
# plt.rcParams['ytick.labelsize'] = 'large'

In [229]:
RANDOM_STATE= 42
BATCH_SIZE = 128
LR = 2e-4
EPOCHS = 5
# MAX_LEN
# tf-idf max feature
# TFIDF_MAXFEATURE=60000 # ถ้าเยอะมาก ต้องจำกัด
# MIN_DF=2

# MLP Config
HIDDEN_DIM=300
MAX_LEN = 256

MAX_VOCAB=20000
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
PAD_ID = 0
UNK_ID = 1

<a id='IMDB'></a>
# IMDB dataset
We retrieve from [Kaggle](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews?resource=download) the csv file "IMDB Dataset.csv" consisting of 50'000 IMDB movies and TV shows reviews with their positive or negative sentiment classification.

In [230]:
# Storing the csv file into a DataFrame "df"

df = pd.read_csv('../input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv')
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


We print the basic properties of the DataFrame. In particular, we note that there are no null values in the DataFrame.

In [231]:
df.sentiment = [1 if s == 'positive' else 0 for s in df.sentiment]
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1
...,...,...
49995,I thought this movie did a down right good job...,1
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",0
49997,I am a Catholic taught in parochial elementary...,0
49998,I'm going to have to disagree with the previou...,0


<a id='preprocessing'></a>
# Data preprocessing
First, we use regular expressions to make the following transformations to the reviews:

- remove punctuation marks
- remove HTML tags
- remove URL's
- remove characters which are not letters or digits
- remove successive whitespaces
- convert the text to lower case
- strip whitespaces from the beginning and the end of the reviews

In [232]:
# Storing in "before_process" a random example of review before preprocessing
# Defining and applying the function "process" performing the transformations of the reviews
# Storing in "after_process" the example of review after preprocessing

idx = random.randint(0, len(df)-1)
before_process = df.iloc[idx][0]

def process(x):
    x = re.sub('[,\.!?:()"]', '', x)
    x = re.sub('<.*?>', ' ', x)
    x = re.sub('http\S+', ' ', x)
    x = re.sub('[^a-zA-Z0-9]', ' ', x)
    x = re.sub('\s+', ' ', x)
    return x.lower().strip()

df['review'] = df['review'].apply(lambda x: process(x))
after_process = df.iloc[idx][0]
after_process

<>:9: SyntaxWarning: invalid escape sequence '\.'
<>:11: SyntaxWarning: invalid escape sequence '\S'
<>:13: SyntaxWarning: invalid escape sequence '\s'
<>:9: SyntaxWarning: invalid escape sequence '\.'
<>:11: SyntaxWarning: invalid escape sequence '\S'
<>:13: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_55/4004618312.py:9: SyntaxWarning: invalid escape sequence '\.'
  x = re.sub('[,\.!?:()"]', '', x)
/tmp/ipykernel_55/4004618312.py:11: SyntaxWarning: invalid escape sequence '\S'
  x = re.sub('http\S+', ' ', x)
/tmp/ipykernel_55/4004618312.py:13: SyntaxWarning: invalid escape sequence '\s'
  x = re.sub('\s+', ' ', x)
/tmp/ipykernel_55/4004618312.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  before_process = df.iloc[idx][0]
/tmp/ipykernel_55/4004618312.py:17: FutureWarning: Series.__

'i remember having a pretty low regard for a venture like this when it was first released james not jim belushi a hammy kid actress and a cheesy title in a john hughes formula you couldn t have paid me to see it 15 years ago but i got caught up watching it while wasting away a sunday afternoon and it hits me on a couple of levels the fairy tale part pretty woman part reverse pretty woman the very vulnerable elizabeth perkins in miracle on 42nd street like performance by kelly lynch the escapism over all it gently pulls some very nice strings it s pretty hard not to fall into the story develop a crush on kelly lynch identify with james belushi dislike the stiff bad guy boyfriend and laugh at the curley sue lines has all the ups and downs with a happy ending and the kind of message you want to hear go ahead waste your time on this movie it s worth it'

Next, we remove stopwords from the reviews using the [word_tokenize()](https://www.nltk.org/_modules/nltk/tokenize.html#word_tokenize) function from the [nltk.tokenize]((https://www.nltk.org/api/nltk.tokenize.html) package.

In [233]:
# Storing in "sw_set" the set of English stopwords provided by nltk
# Defining and applying the function "sw_remove" which remove stopwords from reviews
# Storing in "after_removal" the example of review after removal of the stopwords

sw_set = set(nltk.corpus.stopwords.words('english'))

def tokenize(text):
    return nltk.tokenize.word_tokenize(text)
    
def sw_remove(x):
    words = tokenize(x.lower())
    filtered_list = [word for word in words if word not in sw_set]
    return filtered_list

df['review'] = df['review'].apply(lambda x: sw_remove(x))
after_removal = sw_remove(after_process)
after_removal

['remember',
 'pretty',
 'low',
 'regard',
 'venture',
 'like',
 'first',
 'released',
 'james',
 'jim',
 'belushi',
 'hammy',
 'kid',
 'actress',
 'cheesy',
 'title',
 'john',
 'hughes',
 'formula',
 'paid',
 'see',
 '15',
 'years',
 'ago',
 'got',
 'caught',
 'watching',
 'wasting',
 'away',
 'sunday',
 'afternoon',
 'hits',
 'couple',
 'levels',
 'fairy',
 'tale',
 'part',
 'pretty',
 'woman',
 'part',
 'reverse',
 'pretty',
 'woman',
 'vulnerable',
 'elizabeth',
 'perkins',
 'miracle',
 '42nd',
 'street',
 'like',
 'performance',
 'kelly',
 'lynch',
 'escapism',
 'gently',
 'pulls',
 'nice',
 'strings',
 'pretty',
 'hard',
 'fall',
 'story',
 'develop',
 'crush',
 'kelly',
 'lynch',
 'identify',
 'james',
 'belushi',
 'dislike',
 'stiff',
 'bad',
 'guy',
 'boyfriend',
 'laugh',
 'curley',
 'sue',
 'lines',
 'ups',
 'downs',
 'happy',
 'ending',
 'kind',
 'message',
 'want',
 'hear',
 'go',
 'ahead',
 'waste',
 'time',
 'movie',
 'worth']

<a id='splitting'></a>
# Data splitting and tokenization
We start by splitting our DataFrame into a training and test lists. We use the [train_test_split()](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) function from the [sklearn.model_selection](https://scikit-learn.org/stable/modules/classes.html#module-sklearn.model_selection) module which allow to perform the splitting randomly with respect to the index of the DataFrame.

In [234]:
from sklearn.model_selection import train_test_split

# df tokenized success with nltk
train_rev, tmp_rev, train_sent, tmp_sent = train_test_split(df['review'], df['sentiment'], test_size=0.1, random_state=RANDOM_STATE)
test_rev, val_rev, test_sent, val_sent = train_test_split(tmp_rev, tmp_sent, test_size=0.5, random_state=RANDOM_STATE)


print('\033[1m' + 'train_rev.shape:' + '\033[0m', train_rev)
print('\033[1m' + 'train_rev.shape:' + '\033[0m', train_rev.shape)
print('\033[1m' + 'test_rev.shape:' + '\033[0m', test_rev.shape)
print('\033[1m' + 'val_sent.shape:' + '\033[0m', val_sent.shape)
print('\033[1m' + 'val_sent.shape:' + '\033[0m', val_sent)

train_rev.shape: 40877    [recently, started, watching, show, say, reall...
18057    [return, jedi, often, remembered, wrong, rathe...
19066    [remember, loved, movie, came, 12, years, old,...
20525    [know, last, reviewer, talking, show, pure, en...
5847     [beginning, excited, see, movie, poster, possi...
                               ...                        
11284    [shadow, magic, recaptures, joy, amazement, fi...
44732    [found, movie, quite, enjoyable, fairly, enter...
38158    [avoid, one, terrible, movie, exciting, pointl...
860      [production, quite, surprise, absolutely, love...
15795    [decent, movie, although, little, bit, short, ...
Name: review, Length: 45000, dtype: object
train_rev.shape: (45000,)
test_rev.shape: (2500,)
val_sent.shape: (2500,)
val_sent.shape: 34234    1
28241    0
1226     1
27004    0
35839    1
        ..
26139    1
3222     0
21111    1
42730    1
25326    0
Name: sentiment, Length: 2500, dtype: int64


#**Text to Vector by word2vec**

In [235]:
import multiprocessing
print("CPU cores:", multiprocessing.cpu_count())
CPU_CORES=4

CPU cores: 4


In [236]:
counter = Counter()
for t in train_rev:
    counter.update(t)

vocab = {PAD_TOKEN: PAD_ID, UNK_TOKEN: UNK_ID}
for i, (w, _) in enumerate(counter.most_common(MAX_VOCAB - 2), start=2):
    vocab[w] = i

def encode(text):
    ids = [vocab.get(t, vocab[UNK_TOKEN]) for t in text][:MAX_LEN]
    if len(ids) < MAX_LEN:
       ids = ids + [PAD_ID] * (MAX_LEN - len(ids))
    return ids

vocab_size = len(vocab)
vocab_size

20000

In [237]:
tokenized_train = train_rev.tolist() # pandas Series -> list
vectorize_model = Word2Vec(
    sentences=tokenized_train, 
    vector_size=HIDDEN_DIM, # embedding size
    window=5,
    min_count=1,
    workers=CPU_CORES,
    sg=1 # 0 = CBOW, 1 = Skip-gram
)

print(type(tokenized_train))
print(type(tokenized_train[0]))
print(tokenized_train[0][:10])

<class 'list'>
<class 'list'>
['recently', 'started', 'watching', 'show', 'say', 'really', 'made', 'laugh', 'appreciate', 'unrealistic']


In [238]:
embedding_weight = np.zeros((vocab_size, HIDDEN_DIM), dtype=np.float32)

# ตารางคำศัพท์ → เวกเตอร์
# PAD row (0) = 0 
# UNK row (1) สุ่มเล็กน้อย
rng = np.random.default_rng(RANDOM_STATE)
embedding_weight[UNK_ID] = rng.normal(0, 0.01, size=(HIDDEN_DIM,)).astype(np.float32)

for word, idx in vocab.items():
    if word in (PAD_TOKEN, UNK_TOKEN):
        continue
    if word in vectorize_model.wv:
        embedding_weight[idx] = vectorize_model.wv[word]

In [239]:
# def masked_mean_pooling(tokens, model): 
#     if MAX_LEN is not None:
#         tokens = tokens[:MAX_LEN]
#     attn_mask = np.ones(len(tokens), dtype=np.float32)

#     if len(tokens) < MAX_LEN:
#         pad_len = MAX_LEN - len(tokens)
#         tokens = tokens + [PAD_ID] * pad_len
#         attn_mask = np.concatenate([attn_mask, np.zeros(pad_len, dtype=np.float32)])

#     # build (T,D) vectors ให้ตรงตำแหน่งกับ mask
#     embedding_matrix = np.zeros((MAX_LEN, HIDDEN_DIM), dtype=np.float32)
#     for i, w in enumerate(tokens):
#         if w in model.wv:
#             embedding_matrix[i] = model.wv[w]

#     denom = attn_mask.sum()
#     if denom == 0:
#         return np.zeros(HIDDEN_DIM, dtype=np.float32)

#     embedding_matrix *= attn_mask[:, None]
#     avg_vectors =(embedding_matrix.sum(axis=0) / denom)
#     return avg_vectors # mean sentence vectors


In [240]:
# text to vector
y_train = (train_sent).astype(np.int64).to_numpy()
y_val = (val_sent).astype(np.int64).to_numpy()
y_test = (test_sent).astype(np.int64).to_numpy()

X_train = train_rev
X_val = val_rev
X_test = test_rev

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:", X_val.shape, "y_val:", y_val.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)

X_train: (45000,) y_train: (45000,)
X_val: (2500,) y_val: (2500,)
X_test: (2500,) y_test: (2500,)


**# MLP Part**

In [241]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [242]:
class ImdbDataset(Dataset):
    def __init__(self, X, y):
        self.X = X.tolist() # list of list[str]
        self.y = y.astype(np.int64).to_numpy()

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        ids = encode(self.X[idx])   # list[int] length MAX_LEN
        return torch.tensor(ids, dtype=torch.long), torch.tensor(self.y[idx], dtype=torch.long)

In [243]:
class MLPClassifier(torch.nn.Module):
    def __init__(self, input_dim, embedding_weight=None, output_dim: int = 2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, HIDDEN_DIM, padding_idx=0)
        if embedding_weight is not None:
            self.embedding.weight.data.copy_(torch.tensor(embedding_weight, dtype=torch.float32))
        self.embedding.weight.requires_grad = True
        self.layers = nn.Sequential(
            nn.Linear(input_dim, HIDDEN_DIM),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(HIDDEN_DIM, HIDDEN_DIM // 2),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(HIDDEN_DIM //2, HIDDEN_DIM // 4),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(HIDDEN_DIM // 4, output_dim)
        )

    def forward(self, input_ids):  # (B,T)
        emb = self.embedding(input_ids)                # (B,T,D)
        mask = (input_ids != PAD_ID).unsqueeze(-1).float()  # (B,T,1)
        sum_vec = (emb * mask).sum(dim=1)              # (B,D)
        count = mask.sum(dim=1).clamp(min=1)           # (B,1)
        mean_vec = sum_vec / count                     # (B,D)
        return self.layers(mean_vec)

In [245]:
model = MLPClassifier(input_dim=HIDDEN_DIM, embedding_weight=embedding_weight).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

In [246]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total = 0.0, 0

    all_preds = []
    all_labels = []

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        logits = model(x)
        loss = criterion(logits, y)

        total_loss += loss.item() * y.size(0)
        total += y.size(0)

        preds = logits.argmax(dim=1)

        all_preds.append(preds.detach().cpu().numpy())
        all_labels.append(y.detach().cpu().numpy())

    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_labels)

    acc = (y_pred == y_true).mean()
    f1 = f1_score(y_true, y_pred, average="binary")  # label 0/1

    return total_loss / total, acc, f1

In [247]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * y.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)

    return running_loss / total, correct / total

In [248]:
def fit(model, train_loader, val_loader, optimizer, criterion, device, epochs):
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "val_f1": []}

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc, val_f1 = evaluate(model, val_loader, criterion, device)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(
            f"Epoch {epoch:02d} | "
            f"train_loss={train_loss:.4f} train_acc={train_acc*100:.2f}% | "
            f"val_loss={val_loss:.4f} val_acc={val_acc*100:.2f}% val_f1={val_f1:.4f}"
        )

    return history

In [249]:
def test(model, test_loader, criterion, device):
    test_loss, test_acc, test_f1 = evaluate(model, test_loader, criterion, device)
    print(f"TEST | loss={test_loss:.4f} acc={test_acc*100:.2f}% f1={test_f1:.4f}")
    return test_loss, test_acc, test_f1

In [250]:
train_ds = ImdbDataset(X_train, train_sent)
val_ds   = ImdbDataset(X_val, val_sent)
test_ds  = ImdbDataset(X_test, test_sent)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

# Train and Test data

In [251]:
print("vocab_size =", len(vocab))
print(list(vocab.items())[:20])
sample_tokens = train_rev.iloc[0][:30]    
print(sample_tokens)
print([w in vocab for w in sample_tokens[:10]])
print("encoded:", encode(train_rev.iloc[0])[:30])

x, y = next(iter(train_loader))
print(x[0][:30])
print("nonpad:", (x[0] != 0).sum().item())
print("unk:", (x[0] == 1).sum().item())

vocab_size = 20000
[('<pad>', 0), ('<unk>', 1), ('movie', 2), ('film', 3), ('one', 4), ('like', 5), ('good', 6), ('time', 7), ('even', 8), ('would', 9), ('really', 10), ('story', 11), ('see', 12), ('well', 13), ('much', 14), ('bad', 15), ('get', 16), ('people', 17), ('great', 18), ('also', 19)]
['recently', 'started', 'watching', 'show', 'say', 'really', 'made', 'laugh', 'appreciate', 'unrealistic', 'aspects', 'along', 'everything', 'else', 'people', 'said', 'show', 'realistic', 'reactions', 'dead', 'among', 'things', 'going', 'accept', 'ned', 'bring', 'dead', 'back', 'life', 'accept']
[True, True, True, True, True, True, True, True, True, True]
encoded: [880, 515, 55, 39, 49, 10, 21, 307, 971, 1944, 1280, 228, 155, 210, 17, 193, 39, 658, 3609, 230, 637, 79, 71, 1564, 3879, 596, 230, 53, 37, 1564]
tensor([  264,   227,   401,     4,   197,   521,   120,    12, 17977,  6485,
            1,  2832,  1062,   702,   127,   677,   624,  1052,   428,    31,
         3484,    12,   301,   134,

In [252]:
history = fit(model, train_loader, val_loader, optimizer, criterion, DEVICE, epochs=EPOCHS)

Epoch 01 | train_loss=0.4340 train_acc=80.09% | val_loss=0.2750 val_acc=88.68% val_f1=0.8877
Epoch 02 | train_loss=0.2543 train_acc=90.00% | val_loss=0.2484 val_acc=90.56% val_f1=0.9078
Epoch 03 | train_loss=0.2154 train_acc=91.84% | val_loss=0.2384 val_acc=90.76% val_f1=0.9094
Epoch 04 | train_loss=0.1859 train_acc=93.34% | val_loss=0.2382 val_acc=91.24% val_f1=0.9144
Epoch 05 | train_loss=0.1603 train_acc=94.43% | val_loss=0.2904 val_acc=88.68% val_f1=0.8958


In [253]:
test_loss, test_acc,test_f1 = test(model, test_loader, criterion, DEVICE)

TEST | loss=0.3216 acc=87.72% f1=0.8866


In [269]:
@torch.no_grad()
def predict_text(model, text, device):
    model.eval()

    # preprocess เหมือน train
    text = process(text)
    text = sw_remove(text)
    encode_text = encode(text)
    x = torch.tensor([encode_text], dtype=torch.long, device=device)
    logits = model(x)
    probs = torch.softmax(logits, dim=1).squeeze(0)  # (2,)

    pred_id = int(torch.argmax(probs).item())
    confidence = float(probs[pred_id].item()) * 100

    label = "positive" if pred_id == 1 else "negative"
    return label, confidence, probs.detach().cpu().numpy()

In [270]:
label, conf, probs = predict_text(model,"This movie was amazing and fun!5555", DEVICE)
print("result:",label, f"{conf:.8f}%")

result: positive 99.99896288%
